In [1]:
import chess 
from chess import pgn
import os
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np


In [ ]:
# hàm để load các file pgn
def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
    return games

files = [file for file in os.listdir("data") if file.endswith(".pgn")]
LIMIT_OF_FILES = len(files) #load all files
game = []
i = 1
for file in tqdm(files):
    game.extend(load_pgn(f"data/{file}"))
    if i >= LIMIT_OF_FILES:
        break
    i += 1
print("Games loaded:", len(game))

 93%|█████████▎| 14/15 [03:50<00:16, 16.44s/it]

Games loaded: 61363


In [2]:
from chess import Board
import numpy as np

In [3]:
def board_to_matrix(board: Board):
# Bàn cờ được biểu diễn dưới dạng ma trận 3D với 13 kênh
# 12 kênh đầu tiên đại diện cho các loại quân cờ (6 loại cho mỗi màu)
# Kênh thứ 13 đại diện cho các nước đi hợp lệ
    matrix = np.zeros((13, 8, 8))
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[piece_type + piece_color, row, col] = 1
    legal_moves = board.legal_moves
    for move in legal_moves:
        to_square = move.to_square
        row_to, col_to = divmod(to_square, 8)
        matrix[12, row_to, col_to] = 1

    return matrix

def create_input_for_nn(games): # Tạo dữ liệu đầu vào và nhãn từ các ván cờ
    X = []
    y = []
    for game in games:
        board = game.board()
        for move in game.mainline_moves():
            X.append(board_to_matrix(board))
            y.append(move.uci())
            board.push(move)
    return np.array(X, dtype=np.float32), np.array(y)


def encode_moves(moves): # Mã hóa các nước đi thành số nguyên
    move_to_int = {move: idx for idx, move in enumerate(set(moves))}
    return np.array([move_to_int[move] for move in moves], dtype=np.float32), move_to_int

In [5]:
X, y = create_input_for_nn(game)  # Lưu giá trị trả về từ hàm
y_encoded, move_to_int = encode_moves(y)
print("NUM of moves", len(y))  # In ra số lượng nước đi

NUM of moves 5538627


In [ ]:

y, move_to_int = encode_moves(y) # Mã hóa các nước đi thành số nguyên
num_classes = len(move_to_int)

X  = torch.tensor(X, dtype = torch.float32) # Chuyển đổi X thành tensor
y = torch.tensor(y, dtype = torch.long) # Chuyển đổi y thành tensor

In [7]:
from dataset import ChessDataset
from model import ChessModel

In [ ]:
# Create Dataset and DataLoader
from torch.utils.data import DataLoader # type: ignore
dataset = ChessDataset(X, y) # Tạo dataset từ tensor X và y
dataloader = DataLoader(dataset, batch_size=64, shuffle=True) # Tạo DataLoader để load dữ liệu theo batch size 64

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Model Initialization
model = ChessModel(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

Using device: cuda


In [ ]:
import time
num_epochs = 50
for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)  # chuyển dữ liệu sang GPU nếu có
        optimizer.zero_grad() # trả về gradient về 0

        outputs = model(inputs) 

        # Tính loss và backpropagation
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item()
    end_time = time.time()
    epoch_time = end_time - start_time
    minutes: int = int(epoch_time // 60)
    seconds: int = int(epoch_time) - minutes * 60
    print(f'Epoch {epoch + 1 + 50}/{num_epochs + 1 + 50}, Loss: {running_loss / len(dataloader):.4f}, Time: {minutes}m{seconds}s')

100%|██████████| 86542/86542 [09:26<00:00, 152.83it/s]


Epoch 51/101, Loss: 3.2092, Time: 9m26s


100%|██████████| 86542/86542 [09:25<00:00, 153.09it/s]


Epoch 52/101, Loss: 2.5455, Time: 9m25s


100%|██████████| 86542/86542 [09:29<00:00, 152.08it/s]


...

Epoch 98/101, Loss: 1.8519, Time: 9m31s


100%|██████████| 86542/86542 [09:31<00:00, 151.56it/s]


Epoch 99/101, Loss: 1.8510, Time: 9m31s


100%|██████████| 86542/86542 [09:30<00:00, 151.68it/s]

Epoch 100/101, Loss: 1.8500, Time: 9m30s


In [ ]:

# lưu model vào models/
torch.save(model.state_dict(), "models/TORCH_1_100EPOCHS.pth")

In [13]:
import pickle

with open("models/heavy_move_to_int_1", "wb") as file:
    pickle.dump(move_to_int, file)